Ma a nyelvi modellt először hívod meg a saját kódodból, chatablak nélkül. Megnézzük, mi megy át a hálózaton, mit küld vissza a modell, mibe kerül egy hívás, és hol nem szabad megbízni a válaszában.

**Mielőtt elkezded:** a gépeden fusson az **Ollama**, és legyen letöltve a `gemma4:e4b` modell (a terminálban: `ollama pull gemma4:e4b`). A notebook alapból ezt a helyi modellt használja, ingyen. Egy válasz 5–30 másodperc, attól függően, hogy a modell „gondolkodik-e” előtte. Ez a 6. részben kiderül.

A cellák ugyanúgy működnek, mint az első órán: **olvasós** feladatnál előbb tippelsz, aztán futtatsz; **Copilottal** feladatnál a Copilot írja a kódot, te elolvasod, mielőtt elfogadod; az **ellenőrző** cella pedig kiírja, hogy `RENDBEN`, `HIBA` vagy `MÉG NINCS KÉSZ`. A modell válaszait nem a modell ítéli meg, hanem előre megírt feltételek.

Futtasd sorban (**Shift + Enter**). Ha valami furcsán viselkedik: **Restart**, aztán **Run All**.

In [1]:
import json
import re
import time
import urllib.request

import pandas as pd

import kurzus
from kurzus import chat, kerdez, utolso_hasznalat
from kurzus import ellenoriz, ellenoriz_feltetel, ellenoriz_fuggveny

## 1. Miért kódból, és mi az az API?

A chatablak egy beszélgetésre jó. Elemzéshez viszont ennél több kell:

- **megismételhetőség:** ugyanaz a kérdés ugyanúgy megy el holnap is, a következő havi adattal is;
- **saját adat:** a táblázatot a kód teszi a kérdésbe, nem te másolgatod;
- **automatizálás:** 12 hónap, 40 dokumentum, 3 rendszerprompt: egy ciklus végigmegy rajtuk;
- **ellenőrizhetőség:** a válasz egy változóba kerül, és egy script megnézi, igaz-e.

Az **API** (*application programming interface*) az a felület, amelyen egy program szolgáltatást kér egy másiktól. Nálunk a te kódod kérdez, a nyelvi modell szervere válaszol. Ez a szerver lehet a gépeden futó Ollama vagy egy felhőszolgáltató (OpenAI, Google, Anthropic). A kérés és a válasz is **JSON**: olyan szöveg, amely pont úgy néz ki, mint egy Python-szótár.

## 2. Az első hívás, kézzel

Egy LLM-hívás egyetlen HTTP-kérés. A böngésződ is ilyeneket küld, amikor megnyitsz egy oldalt. A különbség annyi, hogy mi **POST** kérést küldünk: a kéréshez adatot is csatolunk, a kérdésünket JSON-ban.

Először összerakjuk a kérést egy Python-szótárban. Két dolog kell bele: melyik modell válaszoljon, és az üzenetek listája. Most egyetlen üzenet van benne, a kérdésünk (`"role": "user"`). A harmadik sor nem kötelező: a `temperature` (hőmérséklet) 0 értékénél a modell mindig a legvalószínűbb következő szót választja, így ugyanarra a kérdésre ugyanazt adja. Ez kell a megismételhetőséghez.

In [2]:
keres_adat = {
    "model": "gemma4:e4b",
    "messages": [
        {"role": "user", "content": "Egy mondatban: mit mér a fogyasztóiár-index?"},
    ],
    "temperature": 0,
}

# Ez a szöveg megy át a hálózaton: a szótár JSON-ként.
print(json.dumps(keres_adat, ensure_ascii=False))

{"model": "gemma4:e4b", "messages": [{"role": "user", "content": "Egy mondatban: mit mér a fogyasztóiár-index?"}], "temperature": 0}


Most elküldjük. A cím a helyi Ollama OpenAI-kompatibilis végpontja: ugyanilyen címe van az OpenAI, a Google Gemini vagy az Anthropic szerverének is, csak más gépen. A fejlécben (*headers*) megmondjuk, hogy JSON-t küldünk, és itt adnánk át az API-kulcsot is (az Ollama nem kér kulcsot, de a mezőt kitöltjük).

A cella kb. 20 másodpercig fut. Ha `Connection refused` hibát kapsz, nem fut az Ollama: indítsd el, és futtasd újra.

In [3]:
cim = "http://localhost:11434/v1/chat/completions"

keres = urllib.request.Request(
    cim,
    data=json.dumps(keres_adat).encode("utf-8"),
    headers={"Content-Type": "application/json", "Authorization": "Bearer ollama"},
)
with urllib.request.urlopen(keres) as http_valasz:
    valasz = json.loads(http_valasz.read())

print(json.dumps(valasz, ensure_ascii=False, indent=2))

{
  "id": "chatcmpl-701",
  "object": "chat.completion",
  "created": 1791495026,
  "model": "gemma4:e4b",
  "system_fingerprint": "fp_ollama",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "A fogyasztóiár-index (FAP) mérje az idővel változó átlagos árat egy meghatározott termékek és szolgáltatások kosárán, ezzel mutatva meg a pénzérték csökkenését vagyis az inflációt.",
        "reasoning": "Here's a thinking process to arrive at the suggested answer:\n\n1.  **Analyze the Request:** The user is asking, \"What does the Consumer Price Index (CPI) measure?\" and specifically requests the answer in *one sentence* (\"Egy mondatban\").\n2.  **Identify the Core Concept (CPI):** The CPI measures the average change over time in the prices paid by urban consumers for a basket of consumer goods and services. In simpler terms, it tracks inflation.\n3.  **Determine the Target Audience/Tone:** The tone should be clear, concise, and authori

Ez a teljes válasz, szintén szótárként. Keresd meg benne:

- a `choices` listát: ennek első eleme (`[0]`) a modell üzenete (`message`), benne a válasz szövegével (`content`);
- a `reasoning` mezőt: a modell „gondolkodását”, amit a válasz előtt leírt (a 6. részben visszatérünk rá);
- a `finish_reason` mezőt: miért hagyta abba (`stop`: befejezte);
- a `usage` szótárt: hány tokenből állt a kérdés (`prompt_tokens`) és a válasz (`completion_tokens`). Erre a 8. részben, a költségnél lesz szükség.

### Feladat 2.1 – olvasás

Olvasd ki a fenti kiírásból, hány tokenből állt a kérdésed (`prompt_tokens`), és írd be a `None` helyére.

In [4]:
tipp_2_1 = None  # a None helyére írd a fenti kiírásból kiolvasott számot

In [5]:
ellenoriz("tipp_2_1", vart=valasz["usage"]["prompt_tokens"])

MÉG NINCS KÉSZ  tipp_2_1


Egy rövid kérdés is 30 körüli token, pedig csak hat szó. Ennek két oka van: a tokenek szódarabok (a magyar szavak gyakran több darabra esnek), és a szerver a kérdés köré még speciális jelzőtokeneket is tesz (ki beszél, hol a vége).

### Feladat 2.2 – Copilottal

Kérd meg a Copilotot (**Ctrl + I**, Mac: **⌘ + I**):

> Vedd ki a `valasz` szótárból a modell válaszának szövegét (choices → első elem → message → content), és tedd a `valasz_szoveg` változóba. Írd ki.

**Olvasd el, mielőtt elfogadod:** hány szögletes zárójel van a kódban, és mindegyik mit választ ki?

In [6]:
# Ide kerül a Copilot kódja

In [7]:
ellenoriz("valasz_szoveg", vart=valasz["choices"][0]["message"]["content"])

MÉG NINCS KÉSZ  valasz_szoveg


Ugyanezt a két lépést (elküldeni, kiolvasni) a notebook későbbi részeiben többször is használjuk, ezért becsomagolom egy függvénybe. Ez ugyanaz a kód, mint fent, csak névvel:

In [8]:
def kuld(keres_adat):
    """Elküldi a kérést a modellnek, és a teljes választ szótárként adja vissza."""
    keres = urllib.request.Request(
        cim,
        data=json.dumps(keres_adat).encode("utf-8"),
        headers={"Content-Type": "application/json", "Authorization": "Bearer ollama"},
    )
    with urllib.request.urlopen(keres) as http_valasz:
        return json.loads(http_valasz.read())

## 3. Ugyanez a `kurzus.py`-ban: nincs benne varázslat

A notebook mellett van egy `kurzus.py` fájl. A félév során ezt használjuk a modell hívására, hogy ne kelljen mindig kézzel összerakni a kérést. Nézzük meg, mi van benne. A `_keres` függvény küldi el a kérést:

In [9]:
import inspect

print(inspect.getsource(kurzus._keres))

def _keres(utvonal, adat):
    """POST-kérés a beállított végpontra; a választ szótárként adja vissza."""
    url = beallitas("LLM_BASE_URL").rstrip("/") + utvonal
    keres = urllib.request.Request(
        url,
        data=json.dumps(adat).encode("utf-8"),
        headers={"Content-Type": "application/json",
                 "Authorization": f"Bearer {beallitas('LLM_API_KEY')}"},
    )
    try:
        with urllib.request.urlopen(keres, timeout=600) as valasz:
            return json.loads(valasz.read())
    except urllib.error.URLError as hiba:
        if "localhost" in url or "127.0.0.1" in url:
            raise RuntimeError(
                f"Nem érem el a helyi modellt ({url}). Fut az Ollama? "
                "Indítsd el az Ollama alkalmazást, vagy a terminálban: ollama serve"
            ) from hiba
        raise



Ugyanaz, amit te is írtál: `urllib.request.Request`, JSON-ba alakított adat, `Content-Type` és `Authorization` fejléc, a válasz `json.loads`-szal szótárrá. Egy különbség van: a címet és a kulcsot nem a kódból veszi, hanem a beállításokból (erről szól a 4. rész).

A `chat` összerakja a kérés szótárát, elküldi, és a válaszból kiveszi azt, amit te a 2.2-ben: `choices[0]["message"]`. A `kerdez` pedig egyetlen kérdésből és egy opcionális rendszerpromptból (`rendszer`) rakja össze az üzenetlistát:

In [10]:
print(inspect.getsource(kurzus.chat))
print(inspect.getsource(kurzus.kerdez))

def chat(uzenetek, eszkozok=None, modell=None, homerseklet=0.0, gondolkodas=None):
    """Elküldi az üzenetlistát a modellnek, és visszaadja a válaszüzenetet (szótár).

    Az `eszkozok` az OpenAI-formátumú eszközleírások listája. A válaszban vagy szöveg
    (`content`), vagy eszközhívás (`tool_calls`) van. A gondolkodó modelleknél a
    `gondolkodas` szabályozza, gondolkodjon-e válasz előtt. A helyi Gemmánál csak a
    "none" számít: kikapcsolja (kb. ötször gyorsabb, de gyengébb válasz).
    """
    adat = {"model": modell or beallitas("LLM_MODEL"), "messages": uzenetek,
            "temperature": homerseklet, "seed": 42}
    if eszkozok:
        adat["tools"] = eszkozok
    if gondolkodas:
        adat["reasoning_effort"] = gondolkodas
    valasz = _keres("/chat/completions", adat)
    utolso_hasznalat.clear()
    utolso_hasznalat.update(valasz.get("usage", {}))
    return valasz["choices"][0]["message"]

def kerdez(kerdes, rendszer=None, modell=None, gondolkodas=None):
    """Egyetl

### Feladat 3.1 – olvasás

Olvasd el a `kerdez` függvényt. Ha meghívod így: `kerdez("Mi az infláció?", rendszer="Válaszolj röviden.")`, **hány üzenet** lesz a listában, amit elküld a modellnek?

In [11]:
tipp_3_1 = None  # a None helyére írd a tipped

In [12]:
# Ellenőrzés: egy pillanatra kicseréljük a kurzus.chat függvényt egy "kémre",
# amely nem küld el semmit, csak feljegyzi, milyen üzenetlistát kapott.
feljegyzett = []


def kem(uzenetek, **tobbi):
    feljegyzett.append(uzenetek)
    return {"content": ""}


eredeti_chat = kurzus.chat
kurzus.chat = kem
kurzus.kerdez("Mi az infláció?", rendszer="Válaszolj röviden.")
kurzus.chat = eredeti_chat     # visszatesszük az igazit

print(feljegyzett[0])
ellenoriz("tipp_3_1", vart=len(feljegyzett[0]))

[{'role': 'system', 'content': 'Válaszolj röviden.'}, {'role': 'user', 'content': 'Mi az infláció?'}]
MÉG NINCS KÉSZ  tipp_3_1


Mostantól így kérdezünk. Ugyanaz a kérdés, mint a 2. részben, csak egy sorban. A `gondolkodas="none"` kikapcsolja a modell gondolkodását: így pár másodperc az egész (a 6. részben megnézzük, mit veszítünk vele).

In [13]:
print(kerdez("Egy mondatban: mit mér a fogyasztóiár-index?", gondolkodas="none"))
print(utolso_hasznalat)

A fogyasztóiár-index (FPI) méri az átlagos árszint változását, amit a fogyasztók által vásárolt termékek és szolgáltatások árának emelkedése vagy csökkenése jelenti.
{'prompt_tokens': 24, 'completion_tokens': 56, 'total_tokens': 80}


## 4. Szolgáltatófüggetlen kód és az API-kulcs

Mivel a kérés szabványos (OpenAI-kompatibilis), a szolgáltató cserélhető. Ugyanaz a kód megy a gépeden futó Ollamával és a felhőben is. Három dolog változik:

| | helyi Ollama (alapértelmezés) | felhőszolgáltató |
|---|---|---|
| **cím** (`LLM_BASE_URL`) | `http://localhost:11434/v1` | a szolgáltató címe |
| **kulcs** (`LLM_API_KEY`) | nem kell | személyes, titkos kulcs |
| **modell** (`LLM_MODEL`) | `gemma4:e4b` | a szolgáltató modellneve |

A `kurzus.py` ezt a hármat egy `.env` nevű fájlból olvassa, ha van ilyen. A mappában van egy minta, `.env.pelda`:

In [14]:
from pathlib import Path

# A .env.pelda a kurzus.py mellett van.
pelda = Path(kurzus.__file__).resolve().parent / ".env.pelda"
print(pelda.read_text(encoding="utf-8") if pelda.exists() else "Nincs meg a .env.pelda: töltsd le újra a notebookok.zip-et.")

# A notebookok alapból a gépeden futó Ollamát használják, ehhez nem kell semmit beállítanod.
# Ha másik szolgáltatót használnál, másold le ezt a fájlt .env néven, és töltsd ki.
# A .env-et soha ne commitold: API-kulcs van benne.

# Példa: OpenAI
# LLM_BASE_URL=https://api.openai.com/v1
# LLM_API_KEY=ide-jon-a-kulcs
# LLM_MODEL=gpt-4.1-mini
# EMBED_MODEL=text-embedding-3-small

# Példa: Google Gemini (OpenAI-kompatibilis végpont)
# LLM_BASE_URL=https://generativelanguage.googleapis.com/v1beta/openai
# LLM_API_KEY=ide-jon-a-kulcs
# LLM_MODEL=gemini-2.5-flash
# EMBED_MODEL=gemini-embedding-001



**A kulcs olyan, mint egy bankkártya PIN-kódja:** aki megszerzi, a te számládra használja a szolgáltatást. Ezért három szabály:

1. A kulcs **soha nem kerül a kódba**, csak a `.env` fájlba.
2. A `.env` **soha nem kerül fel a GitHubra.** Nézd meg a repód `.gitignore` fájlját: benne van a `.env` sor? Ha nincs, írd bele most.
3. Ha mégis kikerült, azonnal **töröld a kulcsot a szolgáltató oldalán**, és kérj újat. A git-előzményből már nem tudod biztonságosan kivenni.

Az `AGENTS.md`-be is beírjuk, hogy a Copilot se tegye a kulcsot a kódba (lásd a feladat után).

### Feladat 4.1 – hibakeresés

Ez a kód lefut, és működne is egy felhőszolgáltatóval. Mégis nagy baj lesz belőle, amint a repó felkerül a GitHubra.

In [15]:
# Egy hallgató kódja. A kulcs kitalált, de a formája olyan, mint egy igazié.
API_KULCS = "sk-proj-Xq7kitalalt1234567890abcdef"
fejlec = {"Content-Type": "application/json", "Authorization": f"Bearer {API_KULCS}"}

Javítsd ki a lenti cellában: a kulcs a beállításokból jöjjön, a `kurzus.beallitas("LLM_API_KEY")` függvénnyel, és a `kulcs_javitott` változóba kerüljön. (Az ellenőrzés nem írja ki a kulcsot, hogy egy igazi kulcs se kerüljön a notebook kimenetébe.)

In [16]:
# Ide kerül a javítás

In [17]:
ellenoriz_feltetel("kulcs_javitott", lambda kulcs: kulcs == kurzus.beallitas("LLM_API_KEY"),
                   "a kulcs a beállításokból (.env) jön, nem a kódból")

MÉG NINCS KÉSZ  kulcs_javitott


Végül írd be ezt a sort a mappa `AGENTS.md` fájljába, a *Biztonság* alá, hogy a Copilot is tudja:

> - API-kulcsot soha ne írj a kódba. A kulcs a `.env` fájlban van, azt ne nyisd meg és ne commitold.

## 5. Szerepek és a rendszerprompt

Az üzenetlistában minden üzenetnek van szerepe (`role`):

- `system`: a **rendszerprompt**, a beszélgetés egészére szóló utasítás. A modell minden válasznál „látja”.
- `user`: amit a felhasználó kérdez;
- `assistant`: amit a modell korábban válaszolt (a 7. részben lesz fontos).

A rendszerprompt ugyanaz az elv, mint az első óra `AGENTS.md`-je: **ami minden kérdésnél igaz, az ide kerül; ami csak egy feladathoz kell, az nem.**

Kísérlet: ugyanazt a gazdasági kérdést három rendszerprompttal tesszük fel. Hogy gyors legyen, gondolkodás nélkül.

In [18]:
KERDES = "Mekkora a Magyar Nemzeti Bank inflációs célja?"

rendszerpromptok = {
    "nincs": None,
    "rövid": "Válaszolj röviden, magyarul, közgazdász hallgatóknak. Legfeljebb két mondat.",
    "csak szám": "Csak egyetlen számot írj, mértékegység és magyarázat nélkül.",
}

kiserlet = {}
for nev, rendszer in rendszerpromptok.items():
    kiserlet[nev] = kerdez(KERDES, rendszer=rendszer, gondolkodas="none")
    print(f"--- rendszerprompt: {nev}  ({utolso_hasznalat['prompt_tokens']} bemeneti token)")
    print(kiserlet[nev])
    print()

--- rendszerprompt: nincs  (22 bemeneti token)
A Magyar Nemzeti Bank (MNB) **2%-os inflációs célt** tart.

Ez az inflációs cél a bank által meghatározott, és azt jelenti, hogy a MNB törekszik arra, hogy az éves fogyasztói árak emelkedése körülbelül 2% legyen.



--- rendszerprompt: rövid  (55 bemeneti token)
A Magyar Nemzeti Bank (MNB) inflációs célja 2%-os tartományban van. Ez az alacsony, de stabil inflációt célozza meg a pénzpolitikai stabilitás érdekében.



--- rendszerprompt: csak szám  (48 bemeneti token)
2



Figyeld meg a bemeneti tokeneket is: a rendszerprompt is a kérés része, minden hívásnál újra elmegy és fizetsz érte.

A válaszokat nem szemre ítéljük meg, hanem **előre megírt feltételekkel**. Ezeket a kísérlet előtt kell eldönteni: mit várunk az egyes rendszerpromptoktól?

In [19]:
def van_benne_szam(szoveg):
    """Van-e a szövegben legalább egy számjegy."""
    return re.search(r"\d", szoveg) is not None


def csak_szam(szoveg):
    """A szöveg egyetlen szám-e (pl. "3" vagy "2,5"), semmi más."""
    return re.fullmatch(r"\s*-?\d+([.,]\d+)?\s*", szoveg) is not None


for nev, szoveg in kiserlet.items():
    print(f"{nev:10} hossz: {len(szoveg):4} karakter | van benne szám: {van_benne_szam(szoveg)} | csak szám: {csak_szam(szoveg)}")

nincs      hossz:  212 karakter | van benne szám: True | csak szám: False
rövid      hossz:  153 karakter | van benne szám: True | csak szám: False
csak szám  hossz:    1 karakter | van benne szám: True | csak szám: True


### Feladat 5.1 – Copilottal

Írass a Copilottal egy negyedik változatot, egy saját rendszerprompttal:

> Kérdezd meg a `KERDES`-t a `kerdez` függvénnyel, ezzel a rendszerprompttal: „Válaszolj egyetlen magyar mondatban, és a mondat végén zárójelben írd meg, honnan származik az adat.” Használd a `gondolkodas="none"` beállítást. A választ tedd a `valasz_forrassal` változóba, és írd ki.

Az ellenőrzés két előre megírt feltételt néz: van-e a válaszban szám, és van-e benne zárójel.

In [20]:
# Ide kerül a Copilot kódja

In [21]:
ellenoriz_feltetel("valasz_forrassal", lambda v: van_benne_szam(v) and "(" in v and ")" in v,
                   "van benne szám, és zárójelben a forrás")

MÉG NINCS KÉSZ  valasz_forrassal


Figyeld meg, mit írt a zárójelbe. A rendszerprompt forrást kért, és a modell írt is egyet, de nem nézett meg semmit: a forrást is ő fogalmazta. Attól, hogy egy válaszban forrás áll, még nem biztos, hogy onnan származik.

### És igaz is?

A feltételek a válasz **formáját** ellenőrizték, nem azt, hogy igaz-e. Nézzük meg, mit ír az MNB maga. Az Inflációs jelentések elején mindig szerepel a jegybank célja. Az alábbi cella letölti (ha még nincs meg) két jelentést, és kikeresi bennük azt a sort, amelyben a „középtávú cél” szerepel. *A PDF-ek olvasásáról a 3. alkalommal lesz szó; most csak az eredményt nézd.*

In [22]:
import pymupdf  # PDF-olvasó csomag

forrasok = pd.read_csv("data/mnb/forrasok.csv", sep=";")
for jelentes in ["2024-09", "2026-09"]:
    sor = forrasok[forrasok["jelentes"] == jelentes].iloc[0]
    pdf = pymupdf.open(kurzus.letolt(sor["url"], "data/mnb/" + sor["fajl"]))
    for oldalszam, oldal in enumerate(pdf, start=1):
        for szovegsor in oldal.get_text().splitlines():
            if "középtávú cél" in szovegsor:
                print(f"MNB Inflációs jelentés {jelentes}, PDF {oldalszam}. oldal:\n  {szovegsor.strip()}")

print("\nA modell válasza (csak szám):", kiserlet["csak szám"])

MNB Inflációs jelentés 2024-09, PDF 5. oldal:
  A jegybank az inflációs célkövetés rendszerében 2005 augusztusától a 3 százalékos középtávú cél körüli infláció


MNB Inflációs jelentés 2026-09, PDF 5. oldal:
  A jegybank az inflációs célkövetés rendszerében a 2028. januártól hatályos 2,5 százalékos középtávú cél körüli infláció

A modell válasza (csak szám): 2


A 2024. szeptemberi jelentés szerint a cél **3 százalék**; a 2026. szeptemberi szerint **2028 januárjától 2,5 százalék**. Ha a modell 2-t írt (nálam ezt írta), akkor magabiztosan, a kért formában, tévedett. A 2% például az amerikai Fed célja (ezt is az MNB 2026. szeptemberi jelentése írja, a 40. oldalon), nem az MNB-é.

A modell nem olvasta ezeket a jelentéseket, és azt sem tudhatja, mi változott azóta, hogy tanították. A rendszerprompt a formát irányítja, a tudást nem pótolja. A tudást a kontextusba kell tenni: a 3–5. alkalommal ezt csináljuk, a jelentésekből kikeresett részletekkel (RAG).

## 6. A motor belseje: tokenek egy ciklusban

Mi történik a szerveren a kérésünkkel?

1. Az üzenetekből egyetlen hosszú szöveg lesz, speciális jelzőkkel (ki beszél, hol kezdődik a modell válasza).
2. A szöveg **tokenekre** esik: szódarabokra, mindegyiknek van egy sorszáma.
3. A modell ebből **egyetlen következő tokent** jósol. Ezt hozzáfűzi a szöveghez, és jósolja a következőt. Ez egy **ciklus**, amely addig fut, amíg a modell egy speciális **stop-tokent** nem ad, vagy el nem éri a megengedett hosszt.

A modell tehát nem „kitalálja a választ, aztán leírja”: tokenről tokenre írja. Ezért jobb bírálónak, mint alkotónak (erről az előző alkalommal volt szó).

### Feladat 6.1 – olvasás

Ez egy játék-„modell”: egy szótár, amely minden szóhoz megmondja, mi jön utána. Az igazi modell ugyanezt csinálja, csak valószínűségekkel és tokenekkel. Olvasd el a ciklust, és tippeld meg: **hány elem lesz a `szavak` listában** a végén?

In [23]:
tipp_6_1 = None  # a None helyére írd a tipped

In [24]:
# Olvasd el, és csak a tipped beírása után futtasd!
kovetkezo = {
    "<kezdet>": "Az",
    "Az": "infláció",
    "infláció": "2024-ben",
    "2024-ben": "mérséklődött",
    "mérséklődött": ".",
    ".": "<stop>",
}

szavak = []
szo = "<kezdet>"
while True:
    szo = kovetkezo[szo]
    if szo == "<stop>":
        break
    szavak.append(szo)

print(" ".join(szavak))
ellenoriz("tipp_6_1", vart=len(szavak))

Az infláció 2024-ben mérséklődött .
MÉG NINCS KÉSZ  tipp_6_1


A `<stop>` nem kerül a listába, a pont viszont igen: neki is van helye a szótárban, ugyanúgy, mint a szavaknak. A 9. alkalommal ugyanez a `while` ciklus tér vissza nagyobban, az ágens körül.

A megengedett hosszt mi is beállíthatjuk (`max_tokens`). Ha a modell eléri, a ciklus a mondat közepén áll meg, és a `finish_reason` nem `stop` lesz, hanem `length`:

In [25]:
rovid_keres = {
    "model": "gemma4:e4b",
    "messages": [{"role": "user", "content": "Mi az infláció? Két mondatban."}],
    "max_tokens": 8,               # legfeljebb 8 kimeneti token
    "reasoning_effort": "none",    # gondolkodás nélkül
}
eredmeny = kuld(rovid_keres)
print(eredmeny["choices"][0]["message"]["content"])
print("finish_reason:", eredmeny["choices"][0]["finish_reason"])

Az **infláció** a termékek
finish_reason: length


### Gondolkodás a válasz előtt

A `gemma4:e4b` alapból **gondolkodik**, mielőtt válaszol: tokenről tokenre leír egy belső jegyzetet (a 2. rész válaszában ez volt a `reasoning`), és csak utána a választ. A gondolkodás is token, és ugyanabban a ciklusban készül, a válasz **előtt**. Ha 8 tokenben maximáljuk, de nem kapcsoljuk ki a gondolkodást, a 8 token el is fogy rá, a válasz pedig üres marad:

In [26]:
rovid_keres.pop("reasoning_effort")    # most nem kapcsoljuk ki a gondolkodást
eredmeny = kuld(rovid_keres)
uzenet_8 = eredmeny["choices"][0]["message"]
print("gondolkodás:", repr(uzenet_8.get("reasoning", "")))
print("válasz:     ", repr(uzenet_8["content"]))

gondolkodás: "Here's a thinking"
válasz:      ''


Nézzük meg a 2. rész válaszát: milyen hosszú volt a gondolkodás a válaszhoz képest? (A gondolkodás angolul van, ennek a modellnek ez a szokása.)

In [27]:
uzenet = valasz["choices"][0]["message"]
gondolkodas_szoveg = uzenet.get("reasoning", "")   # felhőszolgáltatónál ez a mező hiányozhat

print("A gondolkodás eleje:", gondolkodas_szoveg[:300], "...")
print()
print("A válasz:", uzenet["content"])

A gondolkodás eleje: Here's a thinking process to arrive at the suggested answer:

1.  **Analyze the Request:** The user is asking, "What does the Consumer Price Index (CPI) measure?" and specifically requests the answer in *one sentence* ("Egy mondatban").
2.  **Identify the Core Concept (CPI):** The CPI measures the a ...

A válasz: A fogyasztóiár-index (FAP) mérje az idővel változó átlagos árat egy meghatározott termékek és szolgáltatások kosárán, ezzel mutatva meg a pénzérték csökkenését vagyis az inflációt.


### Feladat 6.2 – Copilottal

> Számold ki, hányszor hosszabb (karakterben) a `gondolkodas_szoveg`, mint a modell válasza (`uzenet["content"]`). Az eredmény a `hanyszor_hosszabb` változóba kerüljön.

In [28]:
# Ide kerül a Copilot kódja

In [29]:
ellenoriz("hanyszor_hosszabb", vart=len(gondolkodas_szoveg) / len(uzenet["content"]))

MÉG NINCS KÉSZ  hanyszor_hosszabb


A gondolkodásért is fizetsz: a `completion_tokens` a gondolkodás tokenjeit is tartalmazza. És várni is kell rá. Mérjük meg ugyanazon a kérdésen, gondolkodással és nélküle. (A cella kb. fél percig fut.)

In [30]:
kerdes_ar = ("Egy termék ára 2023-ban 1000 Ft volt, 2024-ben 10%-kal, 2025-ben további 10%-kal drágult. "
             "Mennyibe kerül 2025-ben? Egy rövid mondatban válaszolj.")

for mod in [None, "none"]:
    kezdes = time.perf_counter()
    valasz_ar = chat([{"role": "user", "content": kerdes_ar}], gondolkodas=mod)
    ido = time.perf_counter() - kezdes
    cimke = "gondolkodással " if mod is None else "gondolkodás nélkül"
    print(f"{cimke}: {ido:5.1f} mp, {utolso_hasznalat['completion_tokens']:4} kimeneti token | {valasz_ar['content']}")

print("Python:", round(1000 * 1.10 * 1.10), "Ft")

gondolkodással :  18.3 mp,  377 kimeneti token | A termék ára 2025-ben 1210 Ft lesz.


gondolkodás nélkül:   1.6 mp,   20 kimeneti token | 2025-ben a termék ára 1210 Ft lesz.
Python: 1210 Ft


Egy ilyen egyszerű kérdésnél a gondolkodás csak lassít és drágít. Bonyolultabb, többlépéses feladatnál (például az ágenseknél) viszont sokat segíthet. A notebookokban ezért ott kapcsolom ki, ahol a válasz minősége nem számít, vagy a feladat egyszerű.

## 7. A modellnek nincs memóriája

A modell minden kérést úgy kap meg, mintha először látna. Ami a kérésben nincs benne, arról nem tud. A chatfelületek „memóriája” egy trükk: a program minden új kérdésnél **a teljes eddigi beszélgetést** újraküldi. Nálunk ezt a mi kódunk csinálja, egy üzenetlistával.

Írd át a nevet a sajátodra, és futtasd le a két cellát.

In [31]:
nevem = "Bálint"   # írd át a sajátodra

elso_kerdes = {"role": "user", "content": f"Szia! {nevem} vagyok, közgazdász hallgató."}
elso_valasz = chat([elso_kerdes], gondolkodas="none")
print(elso_valasz["content"])

Szia Bálint! Nagyszerű, hogy közgazdász hallgató vagy. Ez egy nagyon érdekes és fontos terület.

Miben segíthetek neked? Például:

* **Tanulmányi kérdésekkel kapcsolatban?** (Pl. mikro- vagy makroökonómia fogalmak, modellök)
* **Közgazdasági aktuális eseményekről beszélgetni szeretnél?**
* **Egy konkrét témában információt keresel?**
* **Vagy csak egy általános beszélgetésre vágyol?**

Mondd el, mit foglalkoztál most a tanulmányaidban! 😊


In [32]:
# Egy új kérés, előzmények nélkül:
valasz_elozmeny_nelkul = kerdez("Mi az én nevem?", gondolkodas="none")
print(valasz_elozmeny_nelkul)

# A magyar ragozás miatt (Anna → Annának) a név utolsó magánhangzóját levágjuk a kereséshez.
nev_toredek = nevem.lower().rstrip("ae")
print("\nBenne van a neved a válaszban?", nev_toredek in valasz_elozmeny_nelkul.lower())

Nincs megadva, hogy mi a neved. Te kell mondanod nekem! 😊

Benne van a neved a válaszban? False


Nem tudja, mert nem kapta meg. A második kérésben csak a kérdés volt, a bemutatkozás nem.

### Feladat 7.1 – Copilottal

> Készíts egy `uzenetek` listát, amelyben sorban benne van: az `elso_kerdes`, a modell első válasza (`elso_valasz`), és egy új felhasználói üzenet: „Mi az én nevem?”. Küldd el a `chat` függvénnyel, `gondolkodas="none"` beállítással, és a válasz szövegét (`content`) tedd a `valasz_elozmennyel` változóba. Írd ki.

**Olvasd el, mielőtt elfogadod:** milyen szerepe (`role`) van a lista második elemének?

In [33]:
# Ide kerül a Copilot kódja

In [34]:
ellenoriz_feltetel("valasz_elozmennyel", lambda v: nev_toredek in v.lower(),
                   "a válaszban szerepel a neved, mert az előzményt is elküldted")

MÉG NINCS KÉSZ  valasz_elozmennyel


A „memória” tehát a mi listánk. Ennek ára van: minden kérdésnél újra elküldöd az egész beszélgetést, ezért a bemeneti tokenek száma (és a költség) a beszélgetés hosszával nő. Ha a 7.1 lefutott, nézd meg a `utolso_hasznalat`-ot: a `prompt_tokens` sokkal nagyobb, mint a „Mi az én nevem?” egymagában. A 9. alkalom ágensénél ezt lépésenként ki is írjuk.

## 8. Tokenek és költség

A felhőszolgáltatók tokenre árazva számláznak, külön áron a **bemenetre** (amit elküldesz: rendszerprompt, előzmények, kérdés, dokumentumok) és a **kimenetre** (amit a modell ír, a gondolkodással együtt). Az árat általában **1 millió tokenre** adják meg, dollárban. A kimenet jellemzően többszörösébe kerül, mint a bemenet.

Előbb egy mérés: mennyi token ugyanaz a mondat angolul és magyarul? A `max_tokens: 1` miatt a modell csak egyetlen tokent ír, így a mérés gyors. Az üres kérés is 10 körüli token: ennyi a szerver jelzőtokenjeinek „rezsije”.

In [35]:
mondatok = {
    "angol": "The central bank raised the base rate because inflation was high.",
    "magyar": "A jegybank megemelte az alapkamatot, mert magas volt az infláció.",
    "üres": "",
}
for nyelv, mondat in mondatok.items():
    meres = kuld({"model": "gemma4:e4b", "max_tokens": 1, "reasoning_effort": "none",
                  "messages": [{"role": "user", "content": mondat}]})
    print(f"{nyelv:7} {len(mondat):3} karakter → {meres['usage']['prompt_tokens']:3} bemeneti token")

angol    65 karakter →  21 bemeneti token


magyar   65 karakter →  29 bemeneti token


üres      0 karakter →   9 bemeneti token


Ugyanaz a tartalom magyarul több token, mert a tokenszótárban kevesebb a magyar szó, így a szavak több darabra esnek. Magyar szöveggel tehát ugyanaz a munka drágább.

Most számoljunk költséget. Az alábbi árak **kitalált, kerek példaárak**; mielőtt igazi döntést hozol, nézd meg a szolgáltató aktuális árlistáját.

In [36]:
AR_BE = 0.50   # USD / 1 millió bemeneti token, kitalált példaár
AR_KI = 2.00   # USD / 1 millió kimeneti token, kitalált példaár

### Feladat 8.1 – Copilottal

> Írj egy `koltseg(prompt_tokens, completion_tokens, ar_be, ar_ki)` függvényt, amely visszaadja egy hívás költségét dollárban. Az `ar_be` és az `ar_ki` az 1 millió tokenre vetített ár.

Olvasd el: elosztja valahol egymillióval? És a bemeneti tokeneket a bemeneti árral szorozza?

In [37]:
# Ide kerül a Copilot kódja

In [38]:
def _koltseg(be, ki, ar_be, ar_ki):
    return (be * ar_be + ki * ar_ki) / 1_000_000


ellenoriz_fuggveny("koltseg", [((1_000_000, 0, AR_BE, AR_KI), _koltseg(1_000_000, 0, AR_BE, AR_KI)),
                               ((0, 1_000_000, AR_BE, AR_KI), _koltseg(0, 1_000_000, AR_BE, AR_KI)),
                               ((30, 270, AR_BE, AR_KI), _koltseg(30, 270, AR_BE, AR_KI))],
                   tures=1e-9)

MÉG NINCS KÉSZ  koltseg()


### Feladat 8.2 – Copilottal

Becsüljük meg egy óra költségét. Tegyük fel, hogy 30 hallgató mindegyike 50 olyan hívást küld, mint a 2. részbeli első kérdésed (a `valasz["usage"]` tokenszámaival, gondolkodással együtt).

> A `koltseg` függvénnyel és a `valasz["usage"]` tokenszámaival számold ki, mennyibe kerülne, ha 30 hallgató egyenként 50 ilyen hívást küldene, `AR_BE` és `AR_KI` áron. Az eredmény az `osztaly_koltseg` változóba kerüljön, dollárban.

In [39]:
# Ide kerül a Copilot kódja

In [40]:
_h = valasz["usage"]
ellenoriz("osztaly_koltseg", vart=30 * 50 * _koltseg(_h["prompt_tokens"], _h["completion_tokens"], AR_BE, AR_KI),
          tures=1e-6)

MÉG NINCS KÉSZ  osztaly_koltseg


Egy ilyen rövid kérdés-válasz fillérekbe kerül, még egy egész csoportnak is. Három dolog viszi fel a költséget, és mindhárommal találkozunk a félév során:

- **gondolkodás:** a kimeneti tokenek nagy része gondolkodás lehet (lásd a 6. részt);
- **hosszú kontextus:** ha dokumentumrészleteket teszünk a kérdés mellé (RAG), a bemenet ezer tokenekre nő;
- **ismétlés:** az ágens minden lépésben újraküldi az egész eddigi beszélgetést (7. rész).

A helyi Ollamánál nincs tokenár: ott a te géped dolgozik, és a te idődbe kerül (a mérés a 6. részben).

## 9. Amikor a modell magabiztosan téved

Adjuk oda a modellnek a valódi adatot: a magyar havi inflációt (HICP, az előző év azonos hónapjához képest, %) 2023 januárjától, az Eurostat adataiból. A teljes táblázatot szövegként betesszük a kérdésbe, és megkérdezzük, mennyi volt a havi értékek átlaga 2023 januárja és 2024 decembere között. Most hagyjuk gondolkodni is, hogy a lehető legjobb esélye legyen. (Kb. fél perc.)

In [41]:
adat = pd.read_csv("data/hicp_osszesen.csv", sep=";")
tablazat = adat.to_csv(sep=";", index=False)

kerdes_atlag = (
    "Itt a magyar havi infláció (HICP, éves változás, %), az Eurostat adatai:\n\n"
    + tablazat
    + "\nMennyi volt a havi értékek átlaga 2023 januárja és 2024 decembere között? "
    + "Csak a számot írd le, két tizedesre."
)
modell_valasza = kerdez(kerdes_atlag)
print("A modell válasza:", modell_valasza)
print("Tokenek:", utolso_hasznalat)

A modell válasza: 9.67
Tokenek: {'prompt_tokens': 512, 'completion_tokens': 754, 'total_tokens': 1266}


Határozott szám, pont a kért formában. De jó-e?

### Feladat 9.1 – Copilottal

> Számold ki az `adat` táblából a 2023-as és 2024-es havi `hicp_ev_ev` értékek átlagát (azokat a sorokat, ahol a `honap` 2023-mal vagy 2024-gyel kezdődik). Az eredmény az `atlag_2023_2024` változóba kerüljön.

Olvasd el: pontosan ezt a két évet választja ki? Hány sor marad a szűrés után (hány hónap van két évben)?

In [42]:
# Ide kerül a Copilot kódja

In [43]:
# A helyes érték a nyers adatból, a modelltől függetlenül.
helyes_atlag = adat[adat["honap"].str[:4].isin(["2023", "2024"])]["hicp_ev_ev"].mean()
ellenoriz("atlag_2023_2024", vart=helyes_atlag)

MÉG NINCS KÉSZ  atlag_2023_2024


Most vessük össze a modell számát a Python számával. Ez nem ellenőrzés a szokott értelemben: csak kiírja, mi történt. (A modell válaszából az utolsó számot vesszük ki.)

In [44]:
szamok = re.findall(r"-?\d+(?:[.,]\d+)?", modell_valasza)
modell_szama = float(szamok[-1].replace(",", ".")) if szamok else None

print(f"A modell szerint: {modell_szama}")
print(f"A Python szerint: {helyes_atlag:.2f}")
if modell_szama is None:
    print("A modell válaszában nem volt szám.")
elif abs(modell_szama - helyes_atlag) <= 0.01:
    print("Most eltalálta. Kérdezz rá egy másik időszakra (például a teljes táblázatra): nem mindig sikerül.")
else:
    print(f"Eltérés: {abs(modell_szama - helyes_atlag):.2f} százalékpont. A modell tévedett, és nem jelezte, hogy bizonytalan.")

A modell szerint: 9.67
A Python szerint: 10.63
Eltérés: 0.96 százalékpont. A modell tévedett, és nem jelezte, hogy bizonytalan.


Nálam a modell gondolkodással is rossz átlagot adott, magabiztosan, a kért két tizedesre. Ez nem véletlen hiba. A modell nem számol, hanem tokenről tokenre szöveget ír: a számjegyeket is jósolja. Egy 24 elemű átlagnál ez könnyen elcsúszik, és a válaszból nem látszik, hogy elcsúszott.

Rövidebb listánál néha sikerül: ha csak 2024 tizenkét hónapját kérdeztem, gondolkodással eltalálta, gondolkodás nélkül nem (3,68-at írt 3,72 helyett). Előre nem tudhatod, melyik eset jön, ezért a számot mindig a Python adja.

*Közgazdász megjegyzés: az Eurostat hivatalos éves átlagos HICP-változása 2023-ra 17,0%, 2024-re 3,7% (`prc_hicp_aind`). Ezeket az árindex éves átlagából számolják, ezért a havi ráták átlaga csak közelíti őket.*

### Feladat 9.2 – a te szavaiddal

Írd le egy-két mondatban, a saját szavaiddal, miért nem szabad a modell számára hagyatkozni, és mit csinálsz helyette. A `None` helyére írd, idézőjelek közé.

In [45]:
magyarazat = None  # a None helyére írd a magyarázatod, idézőjelek közé

In [46]:
ellenoriz_feltetel("magyarazat", lambda s: isinstance(s, str) and len(s.split()) >= 8,
                   "legalább egy teljes mondat (8 szó)")

MÉG NINCS KÉSZ  magyarazat


A helyes munkamegosztás tehát: **a számot a Python adja, a modell csak megfogalmazza.** A Python által számolt értéket beletesszük a kérdésbe, és utána ellenőrizzük, hogy a modell nem írta-e át:

In [47]:
szam_szovegben = f"{helyes_atlag:.2f}".replace(".", ",")   # magyar tizedesvesszővel

mondat = kerdez(
    "Fogalmazd meg ezt a tényt egy tárgyilagos mondatban közgazdász hallgatóknak, a számot változtatás nélkül átvéve: "
    "2023 januárja és 2024 decembere között a magyar HICP-infláció (éves változás) havi értékeinek átlaga "
    f"{szam_szovegben}% volt. Forrás: Eurostat.",
    rendszer="Egyetlen mondatot írj, bevezető, felsorolás és megjegyzés nélkül.",
    gondolkodas="none",
)
print(mondat)
ellenoriz_feltetel("mondat", lambda m: szam_szovegben in m or szam_szovegben.replace(",", ".") in m,
                   f"a mondatban a Python által számolt {szam_szovegben} szerepel")

A magyar HICP-infláció éves változása havi átlaga 2023 januárja és 2024 decembere között 10,63%-ot jelentett (Eurostat).
RENDBEN  mondat: a mondatban a Python által számolt 10,63 szerepel


Ez a kurzus egyik alapszabálya: **a várt értéket script számolja a nyers adatból, nem a modell ellenőrzi saját magát.** A 8. alkalommal továbblépünk: a modell maga hívhat meg egy Python-függvényt, amikor számolnia kell (eszközhasználat).

## 10. Gyakorlat: gazdasági kérdések, saját rendszerprompttal

Most te írod a rendszerpromptot. A cél egy kis „kérdezőgép”: ugyanaz a rendszerprompt több gazdasági kérdésre.

In [48]:
kerdesek = [
    "Mi a különbség a KSH fogyasztóiár-indexe és az Eurostat HICP-je között?",
    "Mit jelent, ha a reálbér csökken?",
    "Miért emeli a jegybank a kamatot, ha magas az infláció?",
]

### Feladat 10.1 – a rendszerprompt

Írd meg a saját rendszerpromptodat a `sajat_rendszer` változóba. Gondold át: mi igaz **minden** kérdésre (nyelv, hossz, közönség, mit tegyen, ha nem biztos egy számban)? Ami csak egy kérdéshez kell, az ne kerüljön bele.

In [49]:
sajat_rendszer = None  # a None helyére írd a rendszerpromptod, idézőjelek közé

In [50]:
ellenoriz_feltetel("sajat_rendszer", lambda s: isinstance(s, str) and len(s) >= 20,
                   "van rendszerprompt (legalább 20 karakter)")

MÉG NINCS KÉSZ  sajat_rendszer


### Feladat 10.2 – Copilottal

> Kérdezd meg a `kerdesek` lista minden kérdését a `kerdez` függvénnyel, a `sajat_rendszer` rendszerprompttal és `gondolkodas="none"` beállítással. Minden kérdés után írd ki a kérdést, a választ, és a válasz hosszát karakterben. A válaszokat gyűjtsd a `valaszok` listába.

Olvasd el: a ciklus minden kérdésnél ugyanazt a rendszerpromptot adja át?

In [51]:
# Ide kerül a Copilot kódja

In [52]:
ellenoriz_feltetel("valaszok", lambda v: len(v) == len(kerdesek) and all(isinstance(x, str) and x.strip() for x in v),
                   "minden kérdésre jött szöveges válasz")

MÉG NINCS KÉSZ  valaszok


Olvasd el a válaszokat úgy, mintha egy hallgatótársad dolgozatát javítanád: van bennük tartalmi hiba, pontatlan fogalom, angol szó? A gyors (gondolkodás nélküli) mód magyarja gyengébb. A modell bírálóként jobb, mint alkotóként, de most te vagy a bíráló.

**Kísérletezz tovább** (ehhez nincs ellenőrzés):

- Futtasd a ciklust rendszerprompt nélkül (`rendszer=None`), és hasonlítsd össze a hosszakat.
- Tegyél a rendszerpromptba egy szabályt: „Minden válasz végén írd le, melyik adatforrásból (KSH, Eurostat, MNB) ellenőrizhető az állítás.” Betartja mindhárom kérdésnél?
- Kérdezz egy számot (például „Mennyi volt a magyar infláció 2024 decemberében?”), és vesd össze a `data/hicp_osszesen.csv`-vel. Kinek hiszel?

## Összefoglalás

- **Az LLM-hívás egy HTTP POST kérés JSON-nel** egy OpenAI-kompatibilis címre. A válasz is JSON: a szöveg a `choices[0]["message"]["content"]`-ben, a tokenszám a `usage`-ben.
- **A `kurzus.py` ugyanezt csinálja**, nincs benne varázslat. A szolgáltató cseréjéhez csak a cím, a kulcs és a modellnév változik, a `.env` fájlban.
- **Az API-kulcs titok:** a `.env`-be kerül, soha nem a kódba és soha nem a GitHubra.
- **A rendszerprompt** minden kérésnél jelen lévő utasítás, mint az `AGENTS.md`. A formát irányítja, a hiányzó tudást nem pótolja.
- **A modell tokenről tokenre ír**, egy ciklusban, amíg stop-tokent nem ad. A gondolkodás is token: lassít és drágít.
- **Nincs memória:** a beszélgetést a mi kódunk tárolja egy üzenetlistában, és minden hívásnál újraküldi.
- **Költség = tokenek × ár**, külön a bemenetre és a kimenetre.
- **És a legfontosabb:** a modell magabiztosan téved a számokban. A számot a Python adja a nyers adatból, a modell legfeljebb megfogalmazza.

**Jövő alkalommal:** dokumentumok előkészítése. Az MNB-jelentésekből szöveg lesz, a szövegből darabok: így tudjuk majd a modellnek odaadni azt, amit most nem tudott.